# Solar filament segmentation: full pipeline

This notebook runs the whole pipeline of this repository, from the competition images to `submission.csv`:

1. **Data**: MAGFiLO images and annotations from the Kaggle competition *Solar Filament Segmentation Challenge 2026*.
2. **Preprocessing**: limb darkening and local background divided out (`preprocess.py`).
3. **Model**: U-Net with an ImageNet-pretrained ResNet-34 encoder and a full-resolution skip (`resnet_unet.py`), trained at 2048x2048 on 768x768 crops with soft labels (fraction of annotators marking each pixel) and BCE + Dice + clDice + Lovasz loss (`main.py`, `train_class.py`, `losses.py`).
4. **Inference**: overlapping 768x768 tiles blended into one probability map (`tiling.py`).
5. **Post-processing**: threshold, direction-aware grouping of pieces (`grouping.py`), minimum size and confidence.
6. **Submission**: one row per filament, `filament_id` = `<image name>_<n>`, `segmentation_rle` = pycocotools RLE counts.

By default it uses the trained weights included in the repository (`deliverables/weights/resnet34_unet_2048.pt`, 49 MB),
so the only extra input is the competition data. Set `RUN_TRAINING = True` to train a model first (on Kaggle or
locally; see section 0 for long runs on Kaggle) and `RUN_VALIDATION = True` to compute its validation PQ. The
submission is always written by the model the notebook ends up with: the included weights, or the newly trained ones.

**Setup** (from the repository folder):
```
pip install -r requirements.txt
```
Put the Kaggle competition files in `MAGFiLO_1.0_Kaggle_2026/` (as downloaded: `train/train_images/`,
`train/MAGFiLO_1.0_Annotations_kaggle2026_train.json`, `test/test_images/`), then run all cells.

In [ ]:
# ---- settings
RUN_TRAINING = False     # True: train a model (main.py) and use it instead of the included weights
TRAIN_CONFIG = "main.py" # "main.py": the settings currently in main.py's config (the newest experiment)
                         # "weights": exactly the settings the included weights were trained with
EPOCHS = None            # new runs: number of epochs (None = main.py's 80); the learning rate decays over all of them
TRAIN_HOURS = 11         # stop training after this many hours (Kaggle sessions end at 12 h); None = no limit
RESUME = False           # True: continue the newest unfinished run (on Kaggle: from a previous version's output
                         # added as input; locally: from checkpoints/) instead of starting a new one
RUN_VALIDATION = False   # True: score the model on the validation split (PQ, as reported in README.md)
SWEEP = False            # with RUN_VALIDATION: also try a range of minimum confidences
WEIGHTS = "deliverables/weights/resnet34_unet_2048.pt"
MIN_CONF = None          # filaments with a lower mean probability are dropped; None = predict.py's 0.72
OUT = "submission.csv"   # on Kaggle written to /kaggle/working/
LIMIT = None             # e.g. 3 for a quick check on the first 3 test images
BRANCH = "main"          # Kaggle: branch of the GitHub repository to clone

## 0. Running on Kaggle

Locally, run this notebook from the repository folder with the competition files in `MAGFiLO_1.0_Kaggle_2026/`; the
setup cell below does nothing. On Kaggle it finds the code and the data and arranges them the way the repository
expects:

- **Data:** add the competition (*Add Input → Competitions*). Folder names are found automatically.
- **Code and weights:** turn on *Internet* in the notebook settings (the repository is cloned from GitHub, branch
  `BRANCH`), or attach the repository as a Kaggle dataset. Training needs Internet in any case, for the ImageNet
  weights of the encoder and to install MLflow.
- **Accelerator:** *GPU T4*. On CPU the 180 test images take much longer, and training is not practical.
- **Outputs** go to `/kaggle/working/`: the submission; with training also `checkpoints/`, `mlflow.db`, the
  exported weights (`<run>_weights.pt`) and `submission_<run>.csv`; with validation `validation_<model>.txt` and
  `probs/<model>/`. These are everything needed to switch the submission and the repository to a new model.

**Training on Kaggle.** Sessions end after 12 hours and interactive sessions stop when idle, so train with
*Save Version → Save & Run All (Commit)*, which runs in the background. To finish within one session, choose
`EPOCHS` from the epoch time shown in the log. Training stops itself after `TRAIN_HOURS`
and keeps the last finished epoch in `checkpoints/<run>_last.pt`. To continue: in a new version, add the previous
version's output as input (*Add Input → Your Work*) and set `RESUME = True`.

In [ ]:
# ---- Kaggle setup: code, weights and data (skipped when not on Kaggle)
import os, sys, glob, shutil, subprocess, importlib.util
ON_KAGGLE = os.path.isdir("/kaggle/input")
WORK = "/kaggle/working" if ON_KAGGLE else "."   # where results are written

if not ON_KAGGLE:
    print("not on Kaggle: running from this folder")
else:
    INPUT = "/kaggle/input"
    REPO = "/tmp/solar-filaments"     # outside /kaggle/working, so only the submission ends up in the output
    REPO_URL = "https://github.com/sharu1993/Solar-Filament-Segmentation-Challenge-2026.git"
    DATA_NAME = "MAGFiLO_1.0_Kaggle_2026"     # the data folder name the code expects

    def first(pattern):
        hits = sorted(glob.glob(os.path.join(INPUT, "**", pattern), recursive=True))
        return hits[0] if hits else None

    # code: the repository attached as a dataset, otherwise a clone from GitHub
    if not os.path.exists(os.path.join(REPO, "preprocess.py")):
        found = first("deliverables/predict.py")
        if found:
            shutil.copytree(os.path.dirname(os.path.dirname(found)), REPO, symlinks=True)
            print("code: copied from", os.path.dirname(os.path.dirname(found)))
        else:
            shutil.rmtree(REPO, ignore_errors=True)
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO])
            if r.returncode != 0 or not os.path.exists(os.path.join(REPO, "preprocess.py")):
                raise RuntimeError(f"Code not found: turn on Internet in the notebook settings (and check BRANCH = "
                                   f"{BRANCH!r}), or attach the repository as a Kaggle dataset.")
            print(f"code: cloned {REPO_URL} ({BRANCH})")
    assert os.path.exists(os.path.join(REPO, "deliverables", "weights", "resnet34_unet_2048.pt")), "weights missing from the code"

    # data: link the competition files into the layout the code expects
    annotations = first("*Annotations*train*.json") or first("*Annotations*.json")
    train_dir, test_dir = first("train_images"), first("test_images")
    for name, path in (("annotation file", annotations), ("train_images", train_dir), ("test_images", test_dir)):
        assert path, f"{name} not found under {INPUT}: add the competition data (Add Input -> Competitions)"
    data = os.path.join(REPO, DATA_NAME)
    links = {os.path.join(data, "train", "train_images"): train_dir,
             os.path.join(data, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json"): annotations,
             os.path.join(data, "test", "test_images"): test_dir}
    for link, target in links.items():
        os.makedirs(os.path.dirname(link), exist_ok=True)
        if os.path.islink(link):
            os.remove(link)
        if not os.path.exists(link):
            os.symlink(target, link)
        print(f"data: {os.path.relpath(link, REPO)} -> {target}")

    # training output: only /kaggle/working survives the session, so checkpoints/ and mlflow.db live there
    os.makedirs(os.path.join(WORK, "checkpoints"), exist_ok=True)
    if RUN_TRAINING and RESUME:
        last = sorted(glob.glob(os.path.join(INPUT, "**", "checkpoints", "*_last.pt"), recursive=True),
                      key=os.path.basename)   # run names start with date and time
        assert last, "RESUME: no checkpoints/*_last.pt under /kaggle/input; add the previous version's output (Add Input -> Your Work)"
        previous = os.path.dirname(os.path.dirname(last[-1]))
        for f in glob.glob(os.path.join(previous, "checkpoints", "*.pt")):
            if not os.path.exists(os.path.join(WORK, "checkpoints", os.path.basename(f))):
                shutil.copy(f, os.path.join(WORK, "checkpoints"))
        if os.path.exists(os.path.join(previous, "mlflow.db")) and not os.path.exists(os.path.join(WORK, "mlflow.db")):
            shutil.copy(os.path.join(previous, "mlflow.db"), WORK)
        print("resume: copied checkpoints and mlflow.db from", previous)
    for name in ("checkpoints", "mlflow.db"):
        if not os.path.lexists(os.path.join(REPO, name)):
            os.symlink(os.path.join(WORK, name), os.path.join(REPO, name))

    missing = [m for m in ["pycocotools"] + (["mlflow"] if RUN_TRAINING else []) if not importlib.util.find_spec(m)]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)

    os.chdir(REPO)
    print("working folder:", os.getcwd())

In [ ]:
import os, sys, glob, json, csv, time, shutil, subprocess, threading
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # torch and OpenCV each ship an OpenMP runtime on macOS
sys.path[:0] = [os.path.abspath("."), os.path.abspath("deliverables")]   # run from the repository folder
import numpy as np, cv2, torch
import matplotlib.pyplot as plt
from pycocotools import mask as mask_utils

import preprocess, tiling, evaluate, predict     # project modules
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
DATA = "MAGFiLO_1.0_Kaggle_2026"
for path in (evaluate.IMAGE_DIR, evaluate.ANNOTATION_FILE, predict.TEST_IMAGES):
    assert os.path.exists(path), f"missing {path}: put the Kaggle competition files in {DATA}/"
print("device:", device, "| torch", torch.__version__)
OUT = os.path.join(WORK, OUT)
if MIN_CONF is None:
    MIN_CONF = predict.MIN_CONF


def run(cmd, hours=None, log=None):
    """Run a script and show its output as it comes (and append it to the file `log`). With `hours`, stop it after
    that long (returns False); training keeps the last finished epoch in checkpoints/<run>_last.pt, so it can be
    resumed later."""
    print("$", " ".join(cmd), flush=True)
    log = open(log, "a") if log else None
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timer = threading.Timer(hours * 3600, proc.terminate) if hours else None
    if timer:
        timer.start()
    for line in proc.stdout:
        print(line, end="", flush=True)
        if log:
            log.write(line)
    proc.wait()
    if log:
        log.close()
    stopped = bool(timer) and not timer.is_alive() and proc.returncode != 0
    if timer:
        timer.cancel()
    if stopped:
        print(f"stopped after {hours} h", flush=True)
        return False
    if proc.returncode != 0:
        raise RuntimeError(f"{cmd[2]} failed (exit code {proc.returncode})")
    return True

## 1. Data

707 training images (2048x2048 grayscale JPEG from the GONG network) with 1,154 annotation sets: 296 images were
annotated by 2 or 3 people independently, and every copy is a complete annotation. 180 unlabelled test images.

In [ ]:
ann = json.load(open(evaluate.ANNOTATION_FILE))
files = {i["file_name"] for i in ann["images"]}
test_paths = sorted(glob.glob(os.path.join(predict.TEST_IMAGES, "*.jpeg")))
print(f"training images: {len(files)}, annotation sets: {len(ann['images'])}, filaments: {len(ann['annotations'])}")
print(f"test images: {len(test_paths)}")

## 2. Preprocessing

The disk is darker towards the edge (limb darkening), and the background brightness varies across the disk and
between images. `preprocess.local_normalise` fits the solar disk, divides out the radial brightness profile and a
local median background, and standardises each image (quiet disk ~0, filaments strongly negative, sky = -5).

In [ ]:
img = cv2.imread(test_paths[0], cv2.IMREAD_GRAYSCALE)
norm = preprocess.local_normalise(img)
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("Raw H-alpha image")
ax[1].imshow(norm, cmap="gray", vmin=-5, vmax=3); ax[1].set_title("After local_normalise")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 3. Training (optional)

Training settings live in the `config` dictionary of `main.py` (logged to MLflow with every run) and are saved in
every checkpoint. `TRAIN_CONFIG = "main.py"` trains with those settings (the newest experiment);
`TRAIN_CONFIG = "weights"` uses `main.py --config-from <weights>` to retrain the included model's configuration
exactly. `RESUME = True` continues the newest unfinished run instead. Training saves the best checkpoint (by validation IoU) in `checkpoints/`;
`deliverables/export_weights.py` keeps only the model weights in half precision. Skipped by default: the included
weights are the exported best model (run `2026-11-14_03-27_5981095`, validation IoU 0.546).

In [ ]:
RUN_NAME = None   # set when a model is trained here
if RUN_TRAINING:
    cmd = [sys.executable, "-u", "main.py"]
    if RESUME:
        last = sorted(glob.glob("checkpoints/*_last.pt"), key=os.path.basename)
        assert last, "RESUME: no checkpoints/*_last.pt to continue from"
        cmd += ["--resume", last[-1]]
    else:
        if TRAIN_CONFIG == "weights":
            cmd += ["--config-from", WEIGHTS]
        if EPOCHS:
            cmd += ["--epochs", str(EPOCHS)]
    finished = run(cmd, TRAIN_HOURS)
    last = sorted(glob.glob("checkpoints/*_last.pt"), key=os.path.basename)
    best = last[-1].replace("_last.pt", ".pt") if last else None
    if best and os.path.exists(best):
        RUN_NAME = os.path.basename(best)[:-3]
        state = torch.load(last[-1], map_location="cpu", weights_only=False)
        print(f"{RUN_NAME}: {state['epoch'] + 1} of {state['config']['epochs']} epochs done, best validation IoU "
              f"{state['best_val_iou']:.4f}" + ("" if finished else
              " (not finished: save this version, then continue it in a new version with RESUME = True)"))
        WEIGHTS = os.path.join(WORK, f"{RUN_NAME}_weights.pt")   # exported best epoch, half precision
        run([sys.executable, "-u", "deliverables/export_weights.py", best, WEIGHTS])
    else:
        print("no finished epoch yet; using", WEIGHTS)
model, config = evaluate.load_model(WEIGHTS, device)
print(f"{WEIGHTS}: model {config['model']}, {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters, "
      f"resolution {config['resize_to']}, tiles {config['tile_size']} (overlap {config['tile_overlap']}), "
      f"preprocessing {config.get('preprocess')}")

## 4. Validation (optional)

`evaluate.py` scores the model on the validation split (20% of the training image files, seed 42, grouped by
file) with the competition's Panoptic Quality, against every annotator copy. With the post-processing below it
reports **PQ 0.440** (SQ 0.678, RQ 0.649; 95% bootstrap CI 0.414-0.462); annotators scored against each other reach 0.345.
The output is saved to `validation_<model>.txt` and the probability maps to `probs/<model>/` (in
`/kaggle/working` on Kaggle). With `SWEEP = True` it also scores a range of minimum confidences (a newly trained model may need a different one
than 0.72; set `MIN_CONF` to the best for the submission).

In [ ]:
if RUN_VALIDATION:
    name = RUN_NAME or os.path.splitext(os.path.basename(WEIGHTS))[0]
    log = os.path.join(WORK, f"validation_{name}.txt")   # the scores, for the README and report
    probs = os.path.join(WORK, "probs", name)             # probability maps, for make_figures.py and re-scoring
    setting = ["--threshold", str(predict.THRESHOLD), "--close", str(predict.CLOSE), "--min-area", str(predict.MIN_AREA)]
    run([sys.executable, "-u", "evaluate.py", "--checkpoint", WEIGHTS, "--save-probs", probs] + setting +
        ["--min-conf", str(MIN_CONF)], log=log)
    if SWEEP:   # re-scores the saved maps, no second pass of the model
        run([sys.executable, "-u", "evaluate.py", "--checkpoint", WEIGHTS, "--load-probs", probs] + setting +
            ["--min-conf"] + [str(c) for c in (0.5, 0.6, 0.65, 0.7, 0.72, 0.75, 0.8, 0.85)], log=log)
    print("validation output saved to", log)

## 5. Inference and post-processing

For each test image: preprocessing as in training, the model on overlapping 768x768 tiles blended into one
2048x2048 probability map, then `predict.filaments`: threshold 0.5, pieces joined into one filament when their ends
face each other across a gap the model partly saw (`grouping.py`), filaments with mean
probability below `MIN_CONF` (0.72 unless set) removed. The submission is written to `OUT`, and for a model
trained here also to `submission_<run>.csv` next to it.

In [ ]:
print(f"threshold {predict.THRESHOLD}, grouping {predict.CLOSE}, min area {predict.MIN_AREA} px, min confidence {MIN_CONF}")
paths = test_paths[:LIMIT]
rows, shown = [], {}
start = time.perf_counter()
for i, path in enumerate(paths, start=1):
    x, (w, h) = predict.load_image(path, config["resize_to"], config.get("preprocess"))
    with torch.no_grad():
        prob = tiling.predict_tiled(model, x.unsqueeze(0).to(device), config["tile_size"], config["tile_overlap"])[0, 0].float().cpu().numpy()
    if prob.shape != (h, w):
        prob = cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)
    name = os.path.splitext(os.path.basename(path))[0]
    masks = predict.filaments(prob, predict.THRESHOLD, predict.CLOSE, predict.MIN_AREA, MIN_CONF)
    rows += [(f"{name}_{n}", predict.rle_counts(m)) for n, m in enumerate(masks, start=1)]
    if len(shown) < 2 and len(masks) >= 3:
        shown[path] = masks
    if i % 20 == 0 or i == len(paths):
        print(f"  {i}/{len(paths)} images, {len(rows)} filaments, {time.perf_counter()-start:.0f}s")
with open(OUT, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filament_id", "segmentation_rle"])
    writer.writerows(rows)
print(f"wrote {len(rows)} filaments ({len(rows)/len(paths):.1f} per image) to {OUT}")
if RUN_NAME:
    named = os.path.join(os.path.dirname(OUT), f"submission_{RUN_NAME}.csv")
    shutil.copy(OUT, named)
    print("and to", named)

## 6. Submission checks

Every row decodes to a 2048x2048 mask, IDs are unique and numbered per image, and the masks of one image do not overlap.

In [ ]:
with open(OUT, newline="") as f:
    reader = csv.reader(f)
    assert next(reader) == ["filament_id", "segmentation_rle"]
    sub = list(reader)
ids = [fid for fid, _ in sub]
assert len(ids) == len(set(ids)), "duplicate filament_id"
per_image = {}
for fid, counts in sub:
    stem, n = fid.rsplit("_", 1)
    m = mask_utils.decode({"size": [2048, 2048], "counts": counts.encode()})
    assert m.shape == (2048, 2048) and m.any()
    per_image.setdefault(stem, []).append((int(n), m))
for stem, items in per_image.items():
    assert sorted(n for n, _ in items) == list(range(1, len(items) + 1))
    assert sum(m.astype(np.int32) for _, m in items).max() <= 1, f"overlapping masks in {stem}"
print(f"OK: {len(sub)} filaments in {len(per_image)} of {len(paths)} images; "
      f"{len(paths) - len(per_image)} images without a predicted filament")

## 7. Example predictions on test images

In [ ]:
fig, axes = plt.subplots(1, max(len(shown), 1), figsize=(8 * max(len(shown), 1), 8), squeeze=False)
for ax, (path, masks) in zip(axes[0], shown.items()):
    ax.imshow(cv2.imread(path, cv2.IMREAD_GRAYSCALE), cmap="gray")
    for m in masks:
        contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        for c in contours:
            ax.plot(c[:, 0, 0], c[:, 0, 1], color="#eb6834", lw=1)
    ax.set_title(f"{os.path.basename(path)}: {len(masks)} predicted filaments"); ax.axis("off")
plt.tight_layout(); plt.show()